# 🧫 Séance 2 — Profilage d'expression génique : ACP, UMAP et clustering

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nevermind78/IA_Biotech/blob/main/notebooks/S2_transcriptomique_clustering.ipynb)

**Module : Intelligence Artificielle et Biotechnologie — Atelier 2 / 5 (2h)**

## 🧬 Le problème biologique

Le **RNA-seq** mesure l'expression de **milliers de gènes** dans un échantillon. Que l'échantillon soit
une **tumeur de patient** ou une **cellule unique**, la question est la même :

> **Peut-on regrouper automatiquement les échantillons qui se ressemblent, *sans connaître leur type
> à l'avance*, puis découvrir quels gènes caractérisent chaque groupe (biomarqueurs) ?**

Nous utilisons le jeu de données de référence **PBMC 3k** (10x Genomics) : **~2 700 cellules
immunitaires du sang** d'un donneur sain, chacune décrite par l'expression de **~33 000 gènes**.
Le sang contient plusieurs populations (lymphocytes T, B, NK, monocytes, plaquettes…) :
l'IA doit les **retrouver seule**. C'est exactement la démarche utilisée pour découvrir des
**sous-types de tumeurs** ou des **réponses cellulaires** à un traitement.

## 🎯 Objectifs de l'atelier
1. Manipuler une matrice **échantillons × gènes** de très grande dimension
2. Faire le **contrôle qualité** et la **normalisation** des comptages RNA-seq
3. Réduire la dimension avec l'**ACP** (PCA) puis **UMAP** pour visualiser en 2D
4. Regrouper les cellules avec **K-Means** et choisir le nombre de groupes
5. Produire une **carte interactive**, isoler un **groupe atypique** et ses **biomarqueurs**

## 🧭 Légende
▶️ exécuter · ✍️ à vous · 🔎 interpréter · 💡 solution (cellule masquée)

⏱️ **Déroulé indicatif :** chargement & QC (25 min) · normalisation (10 min) · ACP (20 min) · UMAP (15 min) · K-Means (20 min) · biomarqueurs & carte interactive (30 min)

---
## 0. Installation et imports ▶️
**Scanpy** est la bibliothèque standard d'analyse single-cell ; nous l'utilisons surtout pour **lire**
les données et faire le contrôle qualité. Les algorithmes d'IA viendront de **Scikit-learn** et **UMAP**.

In [ ]:
# ▶️ Installation (≈ 40 secondes)
%pip install -q scanpy umap-learn

In [ ]:
# ▶️ Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import plotly.express as px
import umap

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sns.set_theme(style="whitegrid")
sc.settings.verbosity = 0
from importlib.metadata import version
print("✅ Environnement prêt — scanpy", version("scanpy"))

---
## 1. Charger la matrice d'expression ▶️

In [ ]:
# ▶️ Téléchargement du jeu PBMC 3k (≈ 6 Mo)
try:
    adata = sc.datasets.pbmc3k()
except Exception as e:
    print("Source principale indisponible :", type(e).__name__, "→ téléchargement depuis 10x Genomics")
    import tarfile, urllib.request
    url = "https://cf.10xgenomics.com/samples/cell-exp/1.1.0/pbmc3k/pbmc3k_filtered_gene_bc_matrices.tar.gz"
    urllib.request.urlretrieve(url, "pbmc3k.tar.gz")
    tarfile.open("pbmc3k.tar.gz").extractall(".")
    adata = sc.read_10x_mtx("filtered_gene_bc_matrices/hg19/", var_names="gene_symbols")

adata.var_names_make_unique()
print(adata)

L'objet `adata` (**AnnData**) est un « tableau Excel augmenté » :
- `adata.X` : la **matrice de comptages** (lignes = cellules, colonnes = gènes)
- `adata.obs` : un tableau Pandas d'informations sur les **cellules** (lignes)
- `adata.var` : un tableau Pandas d'informations sur les **gènes** (colonnes)

In [ ]:
# ▶️ Regardons un petit morceau de la matrice sous forme de DataFrame Pandas
genes_exemple = ["CD3E", "MS4A1", "LYZ", "NKG7", "PPBP", "HBB"]
extrait = pd.DataFrame(adata[:8, genes_exemple].X.toarray(),
                       index=adata.obs_names[:8], columns=genes_exemple)
display(extrait)

nb_zeros = 1 - adata.X.nnz / (adata.n_obs * adata.n_vars)
print(f"Dimensions : {adata.n_obs} cellules × {adata.n_vars} gènes")
print(f"Proportion de zéros dans la matrice : {nb_zeros:.1%}")

### 🔎 Question 1
La matrice contient plus de 97 % de zéros. Proposez deux raisons (biologique et technique) à cela.

*Votre réponse :* …

---
## 2. Contrôle qualité ▶️ ✍️

Trois indicateurs par cellule :
- `n_genes_by_counts` : nombre de gènes détectés. **Trop peu** → cellule vide ou abîmée ;
  **trop** → probablement **deux cellules** capturées ensemble (doublet).
- `total_counts` : nombre total de molécules d'ARN lues.
- `pct_counts_mt` : % d'ARN **mitochondrial**. Une cellule qui meurt perd son ARN cytoplasmique mais
  garde ses mitochondries → **% élevé = cellule morte ou stressée**.

In [ ]:
adata.var["mt"] = adata.var_names.str.startswith("MT-")      # gènes mitochondriaux
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)

qc = adata.obs[["n_genes_by_counts", "total_counts", "pct_counts_mt"]]
display(qc.describe().round(1))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, qc.columns):
    sns.histplot(qc[col], bins=60, ax=ax, color="tab:blue")
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
# ✍️ Choisissez les seuils de filtrage en vous aidant des histogrammes
GENES_MIN = 200     #@param {type:"integer"}
GENES_MAX = 2500    #@param {type:"integer"}
MT_MAX = 5.0        #@param {type:"number"}

garder = ((adata.obs["n_genes_by_counts"] >= GENES_MIN) &
          (adata.obs["n_genes_by_counts"] <= GENES_MAX) &
          (adata.obs["pct_counts_mt"] <= MT_MAX))

plt.figure(figsize=(6, 4.5))
plt.scatter(qc["n_genes_by_counts"], qc["pct_counts_mt"], s=4, c=np.where(garder, "tab:blue", "tab:red"))
plt.axvline(GENES_MIN, ls="--", c="k"); plt.axvline(GENES_MAX, ls="--", c="k"); plt.axhline(MT_MAX, ls="--", c="k")
plt.xlabel("Gènes détectés"); plt.ylabel("% ARN mitochondrial")
plt.title(f"Cellules conservées (bleu) : {garder.sum()} / {adata.n_obs}")
plt.show()

adata = adata[garder].copy()
sc.pp.filter_genes(adata, min_cells=3)          # gènes détectés dans au moins 3 cellules
print(f"Après QC : {adata.n_obs} cellules × {adata.n_vars} gènes")

---
## 3. Normalisation ▶️

Chaque cellule n'est pas séquencée avec la même profondeur. On :
1. ramène chaque cellule à **10 000 comptages** au total (comparaison équitable) ;
2. applique **log(1 + x)** pour réduire l'effet des gènes très exprimés ;
3. garde les **2 000 gènes les plus variables** (les autres apportent surtout du bruit).

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata                       # on garde tous les gènes normalisés pour la recherche de marqueurs

sc.pp.highly_variable_genes(adata, n_top_genes=2000)
print("Gènes très variables retenus :", adata.var["highly_variable"].sum())

# Matrice finale pour l'IA : cellules × 2000 gènes, centrée-réduite
X_hvg = adata[:, adata.var["highly_variable"]].X.toarray()
X_std = np.clip(StandardScaler().fit_transform(X_hvg), -10, 10)
print("Matrice pour l'IA :", X_std.shape)

---
## 4. Analyse en Composantes Principales (ACP) ▶️ 🔎

L'ACP cherche les **directions de plus grande variance** dans l'espace à 2 000 dimensions. Chaque
composante principale (PC) est une **combinaison linéaire de gènes**.

In [ ]:
pca = PCA(n_components=50, random_state=42)
X_pca = pca.fit_transform(X_std)

var_exp = pca.explained_variance_ratio_
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(range(1, 51), var_exp * 100, color="tab:blue")
axes[0].set_xlabel("Composante principale"); axes[0].set_ylabel("% de variance expliquée")
axes[0].set_title("Éboulis des valeurs propres (scree plot)")
axes[1].plot(range(1, 51), np.cumsum(var_exp) * 100, marker=".")
axes[1].set_xlabel("Nombre de composantes"); axes[1].set_ylabel("% de variance cumulée")
axes[1].set_title("Variance cumulée")
plt.tight_layout(); plt.show()

In [ ]:
# ▶️ Projection des cellules sur PC1 et PC2, colorée par l'expression de deux gènes connus
expr = lambda g: adata.raw[:, g].X.toarray().ravel()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, gene in zip(axes, ["LYZ", "CD3E"]):
    pts = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=expr(gene), s=5, cmap="viridis")
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.set_title(f"ACP colorée par {gene}")
    plt.colorbar(pts, ax=ax, label="log-expression")
plt.tight_layout(); plt.show()

In [ ]:
# ▶️ Quels gènes « construisent » PC1 et PC2 ? (coefficients = loadings)
genes_hvg = adata.var_names[adata.var["highly_variable"]]
loadings = pd.DataFrame(pca.components_[:2].T, index=genes_hvg, columns=["PC1", "PC2"])
for pc in ["PC1", "PC2"]:
    tri = loadings[pc].sort_values()
    print(f"{pc}  ⬇ négatifs : {list(tri.index[:8])}")
    print(f"{pc}  ⬆ positifs : {list(tri.index[-8:])}\n")

### 🔎 Question 2
1. Combien de composantes faut-il pour expliquer 25 % de la variance ? Que dit le « coude » de
   l'éboulis ?
2. `LYZ` (lysozyme) est un marqueur des **monocytes**, `CD3E` des **lymphocytes T**. Que sépare PC1
   biologiquement ?

*Votre réponse :* …

---
## 5. UMAP : une carte 2D qui préserve les voisinages ▶️ ✍️

L'ACP est **linéaire** : elle écrase des structures complexes. **UMAP** construit un graphe des plus
proches voisins puis le « déplie » en 2D. On l'applique sur les **premières composantes de l'ACP**
(débruitage + rapidité).

In [ ]:
# ✍️ Paramètres de UMAP : essayez n_voisins = 5, 15, 50 et observez la carte
N_PC = 30         #@param {type:"slider", min:5, max:50, step:5}
N_VOISINS = 15    #@param {type:"slider", min:5, max:100, step:5}
DIST_MIN = 0.3    #@param {type:"slider", min:0.0, max:1.0, step:0.05}

reducteur = umap.UMAP(n_neighbors=N_VOISINS, min_dist=DIST_MIN, random_state=42, n_jobs=1)
X_umap = reducteur.fit_transform(X_pca[:, :N_PC])

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for ax, gene in zip(axes, ["CD3E", "MS4A1", "LYZ"]):
    pts = ax.scatter(X_umap[:, 0], X_umap[:, 1], c=expr(gene), s=4, cmap="viridis")
    ax.set_title(f"UMAP — {gene}"); ax.set_xticks([]); ax.set_yticks([])
    plt.colorbar(pts, ax=ax)
plt.tight_layout(); plt.show()

### 🔎 Question 3
Combien d'« îlots » distincts voyez-vous ? Que se passe-t-il quand `N_VOISINS` est très petit (5) ou
très grand (100) ? ⚠️ Les **distances entre îlots** sur une carte UMAP n'ont pas de sens précis !

*Votre réponse :* …

---
## 6. Clustering K-Means ▶️ ✍️

K-Means répartit les cellules en **K groupes** en minimisant la distance de chaque cellule au centre
de son groupe. Il faut **choisir K** : on s'aide de deux critères.
- **Inertie** (somme des distances au centre) : on cherche un « coude » ;
- **Silhouette** (entre −1 et 1) : plus elle est haute, mieux les groupes sont séparés.

In [ ]:
# ▶️ Tester K de 2 à 14 (≈ 30 secondes)
X_clust = X_pca[:, :N_PC]
ks = range(2, 15)
inerties, silhouettes = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X_clust)
    inerties.append(km.inertia_)
    silhouettes.append(silhouette_score(X_clust, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(ks, inerties, marker="o"); axes[0].set_title("Méthode du coude"); axes[0].set_xlabel("K")
axes[1].plot(ks, silhouettes, marker="o", color="tab:green"); axes[1].set_title("Score de silhouette")
axes[1].set_xlabel("K")
plt.tight_layout(); plt.show()

In [ ]:
# ✍️ Choisissez K (la biologie compte autant que les courbes : combien de types de cellules du sang connaissez-vous ?)
K = 9  #@param {type:"slider", min:2, max:14, step:1}

kmeans = KMeans(n_clusters=K, n_init=20, random_state=42).fit(X_clust)
adata.obs["cluster"] = pd.Categorical(kmeans.labels_.astype(str))
print(adata.obs["cluster"].value_counts().sort_index())

---
## 7. 🗺️ Carte interactive des clusters ▶️
Survolez les points avec la souris : chaque point est une cellule. Vous pouvez **zoomer** (cadre),
**isoler un cluster** (double-clic dans la légende) et exporter l'image.

In [ ]:
carte = pd.DataFrame({
    "UMAP1": X_umap[:, 0], "UMAP2": X_umap[:, 1],
    "cluster": adata.obs["cluster"].astype(str).to_numpy(),
    "gènes_détectés": adata.obs["n_genes_by_counts"].to_numpy(),
    "%_mito": adata.obs["pct_counts_mt"].round(2).to_numpy(),
}, index=adata.obs_names)
for g in ["CD3E", "MS4A1", "LYZ", "NKG7", "PPBP"]:
    carte[g] = expr(g).round(2)

ordre = sorted(carte["cluster"].unique(), key=int)
fig = px.scatter(carte, x="UMAP1", y="UMAP2", color="cluster", category_orders={"cluster": ordre},
                 hover_name=carte.index, hover_data=["gènes_détectés", "%_mito", "CD3E", "MS4A1", "LYZ", "NKG7", "PPBP"],
                 title=f"PBMC 3k — UMAP + K-Means (K = {K})", width=850, height=650)
fig.update_traces(marker=dict(size=4))
fig.show()

In [ ]:
# ▶️ Colorer la carte par n'importe quel gène
def carte_gene(gene):
    if gene not in adata.raw.var_names:
        print("Gène inconnu :", gene); return
    f = px.scatter(carte, x="UMAP1", y="UMAP2", color=expr(gene), color_continuous_scale="Viridis",
                   hover_name=carte.index, title=f"Expression de {gene}", width=700, height=550)
    f.update_traces(marker=dict(size=4)); f.show()

GENE = "NKG7"  #@param ["NKG7", "MS4A1", "CD14", "FCGR3A", "CD8A", "IL7R", "FCER1A", "PPBP", "HBB"] {allow-input: true}
carte_gene(GENE)

---
## 8. Biomarqueurs de chaque cluster ▶️

Pour chaque cluster, on compare l'expression moyenne **dans** le cluster et **hors** du cluster.
Un bon marqueur est **fortement plus exprimé** (log fold-change élevé) et **présent dans la majorité**
des cellules du cluster.

In [ ]:
X_raw = adata.raw.X
noms_genes = adata.raw.var_names

def marqueurs(masque, n=10):
    """Gènes surexprimés dans les cellules sélectionnées (masque booléen) par rapport aux autres."""
    dedans, dehors = X_raw[masque], X_raw[~masque]
    moy_in = np.asarray(dedans.mean(axis=0)).ravel()
    moy_out = np.asarray(dehors.mean(axis=0)).ravel()
    pct_in = np.asarray((dedans > 0).mean(axis=0)).ravel()
    pct_out = np.asarray((dehors > 0).mean(axis=0)).ravel()
    res = pd.DataFrame({"logFC": moy_in - moy_out, "%_cellules_dedans": pct_in * 100,
                        "%_cellules_dehors": pct_out * 100}, index=noms_genes)
    return res[res["%_cellules_dedans"] > 25].sort_values("logFC", ascending=False).head(n).round(2)

top = {c: marqueurs((adata.obs["cluster"] == c).to_numpy(), n=6).index.tolist() for c in ordre}
pd.DataFrame(top, index=[f"marqueur_{i+1}" for i in range(6)]).T

### ✍️ Exercice — Annoter les clusters
Comparez les marqueurs trouvés avec ce tableau de marqueurs connus, puis complétez le dictionnaire
`annotation` ci-dessous (`numéro de cluster → type cellulaire`).

| Type cellulaire | Marqueurs connus |
|---|---|
| Lymphocytes T CD4 | `IL7R`, `CD3E`, `LDHB` |
| Lymphocytes T CD8 | `CD8A`, `CD8B`, `CCL5` |
| Lymphocytes B | `MS4A1`, `CD79A`, `CD79B` |
| Cellules NK | `GNLY`, `NKG7`, `GZMB` |
| Monocytes CD14+ | `CD14`, `LYZ`, `S100A8` |
| Monocytes FCGR3A+ | `FCGR3A`, `MS4A7` |
| Cellules dendritiques | `FCER1A`, `CST3` |
| Plaquettes (mégacaryocytes) | `PPBP`, `PF4` |

In [ ]:
# ✍️ À vous : complétez (les numéros dépendent de votre K-Means !)
annotation = {
    # "0": "Lymphocytes T CD4",
}
print("Clusters annotés :", len(annotation), "/", K)

In [ ]:
#@title 💡 Solution : annotation automatique par score de marqueurs connus
MARQUEURS_CONNUS = {
    "Lymphocytes T CD4": ["IL7R", "CD3E", "LDHB"],
    "Lymphocytes T CD8": ["CD8A", "CD8B", "CCL5"],
    "Lymphocytes B": ["MS4A1", "CD79A", "CD79B"],
    "Cellules NK": ["GNLY", "NKG7", "GZMB"],
    "Monocytes CD14+": ["CD14", "LYZ", "S100A8"],
    "Monocytes FCGR3A+": ["FCGR3A", "MS4A7"],
    "Cellules dendritiques": ["FCER1A", "CST3"],
    "Plaquettes": ["PPBP", "PF4"],
}
scores = pd.DataFrame({
    type_c: [np.asarray(adata.raw[(adata.obs["cluster"] == c).to_numpy(), genes].X.mean(axis=0)).mean()
             for c in ordre]
    for type_c, genes in MARQUEURS_CONNUS.items()}, index=ordre)
# score relatif : on centre chaque type cellulaire pour comparer les clusters entre eux
scores_rel = (scores - scores.mean()) / scores.std()
annotation = scores_rel.idxmax(axis=1).to_dict()
plt.figure(figsize=(9, 5))
sns.heatmap(scores_rel, cmap="RdBu_r", center=0, annot=True, fmt=".1f")
plt.title("Score des marqueurs connus par cluster (centré-réduit)"); plt.ylabel("cluster")
plt.show()
annotation

In [ ]:
# ▶️ Carte annotée
carte["type_cellulaire"] = carte["cluster"].map(annotation).fillna("non annoté")
fig = px.scatter(carte, x="UMAP1", y="UMAP2", color="type_cellulaire", hover_name=carte.index,
                 hover_data=["cluster", "CD3E", "MS4A1", "LYZ", "NKG7", "PPBP"],
                 title="PBMC 3k — populations cellulaires identifiées", width=900, height=650)
fig.update_traces(marker=dict(size=4))
fig.show()

---
## 9. Isoler un groupe atypique ✍️ 🔎

Sur la carte, repérez un **petit îlot isolé** (survolez-le pour lire ses coordonnées UMAP).
Saisissez un rectangle qui l'entoure : le code extrait ces cellules et cherche **leurs biomarqueurs**.

In [ ]:
# ✍️ Coordonnées du rectangle (lisez-les sur la carte interactive)
X_MIN = 0.0   #@param {type:"number"}
X_MAX = 0.0   #@param {type:"number"}
Y_MIN = 0.0   #@param {type:"number"}
Y_MAX = 0.0   #@param {type:"number"}

selection = ((carte["UMAP1"] >= X_MIN) & (carte["UMAP1"] <= X_MAX) &
             (carte["UMAP2"] >= Y_MIN) & (carte["UMAP2"] <= Y_MAX)).to_numpy()
print("Cellules sélectionnées :", selection.sum())
if selection.sum() == 0:
    print("⚠️ Rectangle vide : renseignez X_MIN, X_MAX, Y_MIN, Y_MAX (ou exécutez la solution)")

In [ ]:
#@title 💡 Solution : on sélectionne le plus petit cluster K-Means (population rare)
tailles = adata.obs["cluster"].value_counts()
plus_petit = tailles.idxmin()
selection = (adata.obs["cluster"] == plus_petit).to_numpy()
print(f"Plus petit cluster : n°{plus_petit} ({selection.sum()} cellules sur {adata.n_obs})")

In [ ]:
# ▶️ Visualiser le groupe atypique et afficher ses marqueurs
plt.figure(figsize=(6, 5))
plt.scatter(X_umap[:, 0], X_umap[:, 1], s=3, c="lightgrey")
plt.scatter(X_umap[selection, 0], X_umap[selection, 1], s=8, c="crimson")
plt.title(f"Groupe atypique : {selection.sum()} cellules"); plt.xticks([]); plt.yticks([])
plt.show()

marqueurs_atypiques = marqueurs(selection, n=15)
marqueurs_atypiques

### 🔎 Question 4
Quels sont les gènes marqueurs du groupe isolé ? Cherchez 2 d'entre eux sur
[GeneCards](https://www.genecards.org) ou [UniProt](https://www.uniprot.org). De quel type cellulaire
s'agit-il ? Pourquoi un tel groupe pourrait-il **passer inaperçu** dans une analyse « en masse »
(bulk RNA-seq) d'un échantillon de sang ?

*Votre réponse :* …

---
## 10. 🏁 Livrable : exporter la carte et les résultats ▶️

In [ ]:
fig.write_html("carte_clusters_PBMC.html")                   # carte interactive autonome
carte.to_csv("cellules_clusters.csv")                        # 1 ligne par cellule
marqueurs_atypiques.to_csv("marqueurs_groupe_atypique.csv")
print("✅ Fichiers créés : carte_clusters_PBMC.html, cellules_clusters.csv, marqueurs_groupe_atypique.csv")

try:
    from google.colab import files
    for f in ["carte_clusters_PBMC.html", "cellules_clusters.csv", "marqueurs_groupe_atypique.csv"]:
        files.download(f)
except ImportError:
    print("(Hors Colab : les fichiers sont dans le dossier courant)")

---
## 🚀 Pour aller plus loin (bonus)

**Défi A — Algorithme de clustering de référence.** En single-cell, on utilise plutôt le clustering
**Leiden** sur le graphe des voisins. Exécutez la cellule suivante et comparez avec K-Means grâce à
l'**indice de Rand ajusté** (ARI = 1 : partitions identiques ; 0 : hasard).

In [ ]:
#@title Défi A : Leiden vs K-Means
%pip install -q igraph leidenalg
from sklearn.metrics import adjusted_rand_score
adata.obsm["X_pca"] = X_pca
sc.pp.neighbors(adata, n_neighbors=N_VOISINS, n_pcs=N_PC)
sc.tl.leiden(adata, resolution=0.6, flavor="igraph", n_iterations=2, directed=False, random_state=0)
print("Clusters Leiden :", adata.obs["leiden"].nunique())
print(f"ARI K-Means vs Leiden : {adjusted_rand_score(adata.obs['cluster'], adata.obs['leiden']):.2f}")
carte["leiden"] = adata.obs["leiden"].astype(str).to_numpy()
px.scatter(carte, x="UMAP1", y="UMAP2", color="leiden", width=750, height=550,
           title="Clustering Leiden").update_traces(marker=dict(size=4)).show()

**Défi B — Sensibilité aux choix.** Relancez l'analyse avec `N_PC = 10` et `K = 5`. Les plaquettes
sont-elles encore isolées ? Qu'en concluez-vous sur la **robustesse** d'un clustering ?

---
## 📝 Bilan de la séance

| Étape | Notion clé |
|---|---|
| QC & normalisation | données brutes ≠ données exploitables ; % mitochondrial |
| ACP | réduction **linéaire**, variance expliquée, loadings = gènes |
| UMAP | réduction **non linéaire**, préserve les voisinages (pas les distances globales) |
| K-Means | **apprentissage non supervisé**, choix de K (coude, silhouette, biologie) |
| Marqueurs | du cluster mathématique à l'**interprétation biologique** |

**Livrable à rendre :** la carte `carte_clusters_PBMC.html` + vos réponses aux questions 🔎 1 à 4 et
l'identité du groupe atypique avec ses 5 premiers biomarqueurs.

➡️ **Séance suivante :** les **séquences** de protéines et les **modèles de langage** biologiques.